# Code-Smell Pipeline: Complete Colab Setup

This notebook clones the code-smell-pipeline repository, prepares the dataset, runs validation tests, and performs end-to-end inference.

## Quick Start

1. Upload your dataset JSON/JSONL file to `/content/` (or use an existing one)
2. Update `DATASET_PATH` in the first cell to match your file
3. Run each cell sequentially
4. Watch for clear status messages after each step

## Step 1: Clone Repository and Install Dependencies

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# === CONFIGURATION ===
REPO_URL = "https://github.com/AliceSheena/code-smell-pipeline.git"
REPO_DIR = Path("/content/code-smell-pipeline")
DATASET_PATH = Path("/content/InplaceRefactor_verified.json")  # ← UPDATE THIS PATH
OUTPUT_DIR = REPO_DIR / "data"
SPLITS_DIR = OUTPUT_DIR / "splits"

print("="*70)
print("STEP 1: Clone Repository and Install Dependencies")
print("="*70)

# Clean and clone
if REPO_DIR.exists():
    print(f"Removing existing {REPO_DIR}...")
    subprocess.run(["rm", "-rf", str(REPO_DIR)], check=True)

print(f"Cloning {REPO_URL}...")
subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True, capture_output=True)
print(f"✓ Repository cloned to {REPO_DIR}")

# Change to repo directory
os.chdir(str(REPO_DIR))
print(f"✓ Changed to {os.getcwd()}")

# Install dependencies
print("\nInstalling dependencies from requirements.txt...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("✓ Dependencies installed")

print(f"\n✓ Setup complete. Working directory: {os.getcwd()}")

## Step 2: Verify Dataset File Exists

In [ ]:
print("="*70)
print("STEP 2: Verify Dataset File")
print("="*70)

if not DATASET_PATH.exists():
    print(f"\n❌ ERROR: Dataset file not found at {DATASET_PATH}")
    print(f"\nFiles in /content/:")
    subprocess.run(["ls", "-lh", "/content/"], capture_output=False)
    print("\n⚠️  Please upload your dataset JSON file to /content/ and update DATASET_PATH above.")
    sys.exit(1)

size_mb = DATASET_PATH.stat().st_size / (1024*1024)
print(f"✓ Dataset found: {DATASET_PATH}")
print(f"  Size: {size_mb:.2f} MB")

## Step 3: Prepare Dataset (Single-Cell Execution)

In [ ]:
import json
import subprocess
from pathlib import Path

print("="*70)
print("STEP 3: Prepare Dataset (Data Prep + Verification + Quick Inference)")
print("="*70)

# Ensure cwd is correct for this cell
os.chdir(str(REPO_DIR))
print(f"Current directory: {os.getcwd()}")

# === 3A: Run Data Preparation ===
print("\n--- 3A: Running Dataset Preparation ---")
prep_script = REPO_DIR / "data" / "prepare_dataset.py"
if not prep_script.exists():
    print(f"❌ ERROR: {prep_script} not found")
    sys.exit(1)

try:
    result = subprocess.run(
        [sys.executable, str(prep_script), "--input", str(DATASET_PATH), "--output-dir", str(OUTPUT_DIR)],
        capture_output=True,
        text=True,
        check=True,
        timeout=300
    )
    print(result.stdout)
    if result.stderr:
        print(f"Warnings/Stderr: {result.stderr}")
except subprocess.CalledProcessError as e:
    print(f"❌ Data preparation failed:")
    print(f"Stdout: {e.stdout}")
    print(f"Stderr: {e.stderr}")
    sys.exit(1)

# === 3B: Verify Splits Exist and Have Content ===
print("\n--- 3B: Verifying Data Splits ---")
if not SPLITS_DIR.exists():
    print(f"❌ ERROR: Splits directory {SPLITS_DIR} does not exist")
    sys.exit(1)

required_files = ["train.jsonl", "val.jsonl", "test.jsonl"]
for fname in required_files:
    fpath = SPLITS_DIR / fname
    if not fpath.exists():
        print(f"❌ ERROR: {fname} not found in {SPLITS_DIR}")
        sys.exit(1)
    
    num_lines = sum(1 for _ in open(fpath))
    print(f"✓ {fname}: {num_lines} lines")
    
    if num_lines > 0:
        first_line = open(fpath).readline()
        first_obj = json.loads(first_line)
        print(f"  First entry smell_type: {first_obj.get('smell_type', 'N/A')}")

# === 3C: Check Prepare Report ===
report_path = OUTPUT_DIR / "prepare_report.json"
if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)
    print(f"\nPrepare Report:")
    print(f"  Clean entries: {report.get('clean_count', 'N/A')}")
    print(f"  Dirty entries: {report.get('dirty_count', 'N/A')}")

print("\n✓ Data preparation and verification complete")

## Step 4: Run Smoke Tests

In [ ]:
print("="*70)
print("STEP 4: Run Smoke Tests")
print("="*70)

os.chdir(str(REPO_DIR))

try:
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-v", "tests/test_smoke.py"],
        capture_output=True,
        text=True,
        timeout=60
    )
    print(result.stdout)
    if result.returncode == 0:
        print("\n✓ All smoke tests passed")
    else:
        print(f"\n❌ Some tests failed (exit code: {result.returncode})")
        if result.stderr:
            print(f"Stderr: {result.stderr}")
except Exception as e:
    print(f"⚠️  Could not run pytest: {e}")
    print("Attempting fallback: direct import test")
    try:
        from tests.test_smoke import SmokeTests
        print("✓ Smoke tests module imported successfully")
    except Exception as e2:
        print(f"⚠️  Warning: {e2}")

## Step 5: Quick Localization Inference

In [ ]:
print("="*70)
print("STEP 5: Quick Localization Inference")
print("="*70)

os.chdir(str(REPO_DIR))

test_file = SPLITS_DIR / "test.jsonl"
if not test_file.exists():
    print(f"⚠️  Warning: {test_file} not found, skipping inference")
else:
    print(f"\nRunning main.py quick-infer with {test_file}...")
    
    try:
        result = subprocess.run(
            [sys.executable, "main.py", "quick-infer", "--input-file", str(test_file), "--use-ast"],
            capture_output=True,
            text=True,
            timeout=120
        )
        
        if result.returncode == 0:
            print("\n✓ Quick inference completed successfully")
            if result.stdout:
                print("\nOutput (first 500 chars):")
                print(result.stdout[:500])
        else:
            print(f"⚠️  Inference exited with code {result.returncode}")
            if "No module" in result.stderr or "cannot import" in result.stderr.lower():
                print("Note: This may be due to missing model files (expected in full training).")
                print(f"Error: {result.stderr[:300]}")
            else:
                print(f"Error output:\n{result.stderr}")
    except subprocess.TimeoutExpired:
        print("⚠️  Inference timed out (expected if models need download)")
    except Exception as e:
        print(f"⚠️  Could not run inference: {e}")

## Step 6: Summary and Next Steps

In [ ]:
print("="*70)
print("PIPELINE SUMMARY")
print("="*70)

print(f"""
✓ Repository cloned from {REPO_URL}
✓ Dependencies installed
✓ Dataset prepared from {DATASET_PATH}
✓ Split files generated in {SPLITS_DIR}
✓ Smoke tests executed
✓ Quick inference tested

Next Steps for Full Training:
------
1. If you want to train the localisation model, run:
   python3 localisation/train_localisation.py
   
2. If you want to train the refactoring model, run:
   python3 refactoring/train_codet5.py
   
3. For full evaluation:
   python3 evaluation/run_eval.py

Repository Structure:
------
{REPO_DIR}/
├── data/
│   ├── prepare_dataset.py        (dataset preparation)
│   ├── dataset_utils.py          (data loading utilities)
│   └── splits/
│       ├── train.jsonl           (training data)
│       ├── val.jsonl             (validation data)
│       └── test.jsonl            (test data)
├── localisation/                 (code smell localisation models)
├── refactoring/                  (refactoring code generation)
├── evaluation/                   (evaluation metrics)
├── pipeline/                     (end-to-end inference)
├── requirements.txt              (dependencies)
├── main.py                       (CLI wrapper)
└── localization_config.json      (model config)

GPU Information:
------
GPU is NOT required for data preparation or basic testing.
GPU IS recommended for:
  - Full model training (UnixCoder fine-tuning, CodeT5 fine-tuning)
  - Large batch inference
  
Current GPU status: {subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip() or 'No GPU detected'}

""")

print("="*70)
print("✓ Setup complete! Ready for training or further experimentation.")
print("="*70)